# Modulo 2: Supervised Learning e Ottimizzazione
## Progetto d'Esame: Predizione del PEGI dei Videogiochi di Steam

Questo notebook rappresenta la seconda fase del progetto d'esame. L'obiettivo principale è l'addestramento e l'ottimizzazione di un modello supervisionato di classificazione basato su **XGBoost** per predire la classe PEGI a partire dalle feature cromatiche estratte dal Modulo 1 (spazio colore LAB) e dai metadati di prezzo e genere.

### Fasi implementate in questo Modulo:
1. **Feature Engineering**: Caricamento del dataset delle feature, applicazione di *Multi-Hot Encoding* sui generi dei videogiochi ed allineamento delle etichette PEGI per la compatibilità con XGBoost (mappatura in classi discrete `[0, 1, 2, 3, 4]`).
2. **Validazione e Gestione dello Sbilanciamento**: Configurazione di una *Stratified K-Fold Cross-Validation* a 5 fold per preservare le proporzioni delle classi. Calcolo e applicazione dei pesi campionari per contrastare lo sbilanciamento del dataset.
3. **Ottimizzazione degli Iperparametri con Optuna**: Configurazione di una ricerca sistematica guidata per massimizzare la metrica di **Macro F1-Score**.
4. **Addestramento Finale ed Esportazione**: Addestramento del classificatore XGBoost finale sui migliori iperparametri ed esportazione del modello in formato JSON.
5. **Valutazione e Analisi**: Report di classificazione, matrice di confusione normalizzata in percentuale e analisi della *Feature Importance* per valutare l'impatto dei colori rispetto ai metadati tradizionali.

### 1. Feature Engineering (Cella 1)

Carichiamo il dataset generato precedentemente, codifichiamo la variabile categorica multi-valore `genres` tramite `MultiLabelBinarizer` per creare colonne binarie indicative della presenza del genere (es. `Is_Action`, `Is_Casual`, ecc.), e convertiamo i target PEGI in classi numeriche continue `[0-4]` pronte per XGBoost.

In [ ]:
# Caricamento delle librerie necessarie
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MultiLabelBinarizer
import ast

# 1. Definizione dei percorsi di caricamento (supporto integrato sia per Google Colab che per locale)
colab_path = '/content/steam_features_dataset.csv'
local_path = 'steam_features_dataset.csv'

if os.path.exists(colab_path):
    csv_path = colab_path
    print(f"[INFO] Rilevato ambiente Google Colab. Caricamento da: {csv_path}")
elif os.path.exists(local_path):
    csv_path = local_path
    print(f"[INFO] Rilevato ambiente locale. Caricamento da: {csv_path}")
else:
    csv_path = colab_path
    print(f"[WARNING] Dataset non trovato nei percorsi standard. Uso predefinito di Colab: {csv_path}")

# Lettura del DataFrame
df = pd.read_csv(csv_path)
print(f"[INFO] Righe caricate: {df.shape[0]} | Colonne caricate: {df.shape[1]}")

# 2. Multi-Hot Encoding sulla colonna dei generi ('genres')
# Questa funzione si occupa di convertire le stringhe (o stringhe di liste) in liste Python pulite
def clean_genres(genre_str):
    if pd.isna(genre_str) or not isinstance(genre_str, str):
        return []
    genre_str = genre_str.strip()
    if genre_str.startswith('[') and genre_str.endswith(']'):
        try:
            return ast.literal_eval(genre_str)
        except (ValueError, SyntaxError):
            pass
    return [g.strip() for g in genre_str.split(',') if g.strip()]

df['genres_cleaned'] = df['genres'].apply(clean_genres)

# Inizializzazione e applicazione di MultiLabelBinarizer per creare le colonne binarie
mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(df['genres_cleaned'])
genres_columns = [f"Is_{genere.replace(' ', '_').replace('-', '_').replace('/', '_')}" for genere in mlb.classes_]
genres_df = pd.DataFrame(genres_encoded, columns=genres_columns, index=df.index)

# Concatenazione delle nuove colonne binarie e rimozione delle colonne temporanee o non più necessarie
df_encoded = pd.concat([df.drop(columns=['genres', 'genres_cleaned']), genres_df], axis=1)
print(f"[INFO] Multi-Hot Encoding completato con successo. Generi unici individuati: {len(mlb.classes_)}")

# 3. Conversione del target (pegi / required_age) in classi numeriche discrete da 0 a 4
target_col = 'pegi' if 'pegi' in df_encoded.columns else 'required_age'
print(f"[INFO] Colonna target identificata nel dataset: '{target_col}'")

# Mappatura ufficiale PEGI [3, 7, 12, 16, 18] -> Classi [0, 1, 2, 3, 4]
pegi_mapping = {3: 0, 7: 1, 12: 2, 16: 3, 18: 4}
df_encoded['target_class'] = df_encoded[target_col].map(pegi_mapping)

# Rimozione di eventuali righe con valori target non validi o nulli
if df_encoded['target_class'].isnull().any():
    print("[WARNING] Presenza di target non validi o nulli rilevata. Rimozione delle righe interessate.")
    df_encoded = df_encoded.dropna(subset=['target_class'])

df_encoded['target_class'] = df_encoded['target_class'].astype(int)
df_final = df_encoded.drop(columns=[target_col])

print("[INFO] Associazione Target-Classe completata:")
for pegi_val, class_idx in pegi_mapping.items():
    count = (df_final['target_class'] == class_idx).sum()
    print(f" - PEGI {pegi_val} mappato come Classe {class_idx} (Totale elementi: {count})")

print(f"[INFO] Dimensioni finali del dataset processato: {df_final.shape}")

### 2. Validazione e Gestione dei Pesi (Cella 2)

Configuriamo una procedura di validazione robusta tramite *Stratified K-Fold Cross-Validation* a 5 fold per garantire la stabilità delle stime di generalizzazione. Inoltre, calcoliamo i pesi bilanciati delle classi per compensare lo sbilanciamento del target nativo, definendo un array `sample_weight` per l'addestramento.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from imblearn.combine import SMOTETomek

# 1. Suddivisione delle Feature (X) e del Target (y)
X = df_final.drop(columns=['target_class'])
y = df_final['target_class'].values

print(f"[INFO] Shape delle feature X (input): {X.shape}")
print(f"[INFO] Shape del target y (output): {y.shape}")

# 2. Configurazione di Stratified K-Fold Cross-Validation (5 fold)
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
print(f"[INFO] Stratified K-Fold configurato con successo (n_splits={n_splits}).")

# 3. Inizializzazione di SMOTE + Tomek per il bilanciamento avanzato delle classi
# SMOTETomek combina il sovracampionamento delle classi minoritarie (SMOTE)
# con la rimozione dei link di Tomek (esempi rumorosi sui confini decisionali)
smt = SMOTETomek(random_state=42)
print("[INFO] SMOTE + Tomek Links configurato per l'applicazione locale su ciascun training fold.")

### 3. Ottimizzazione degli Iperparametri con Optuna (Cella 3)

Installiamo ed integriamo la libreria **Optuna** per implementare l'ottimizzazione degli iperparametri di XGBoost. Lo studio ottimizzerà parametri critici quali `max_depth`, `learning_rate`, `n_estimators`, `subsample` ed altri. Per valutare le performance nei fold, massimizzeremo il **Macro F1-Score** per assicurarci che il modello sia accurato anche sulle classi minoritarie.

In [ ]:
# Installazione dei pacchetti richiesti per l'advanced modeling ed ensemble
!pip install -q optuna xgboost lightgbm catboost imbalanced-learn

import optuna
import xgboost as xgb
from sklearn.metrics import f1_score
from imblearn.combine import SMOTETomek
import numpy as np
import warnings

# Ignoriamo i warning per rendere l'output dello studio più pulito e leggibile
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Inizializziamo SMOTETomek per i fold della cross-validation
smt_fold = SMOTETomek(random_state=42)

# Definizione della funzione obiettivo per Optuna
def objective(trial):
    # Spazio di ricerca per gli iperparametri critici di XGBoost
    params = {
        'objective': 'multi:softprob',
        'num_class': 5,
        'eval_metric': 'mlogloss',
        'random_state': 42,
        'n_jobs': -1,
        
        # Iperparametri da ottimizzare
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 350),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 8)
    }
    
    fold_f1_scores = []
    
    # Esecuzione della Cross-Validation a 5 Fold
    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        # Applicazione di SMOTE + Tomek sul solo training fold corrente per evitare data leakage
        try:
            X_train_res, y_train_res = smt_fold.fit_resample(X_train, y_train)
        except Exception:
            # Fallback in caso di classi con pochissimi esempi
            X_train_res, y_train_res = X_train, y_train
        
        # Istanziazione e addestramento del modello
        model = xgb.XGBClassifier(**params)
        model.fit(X_train_res, y_train_res)
        
        # Predizione
        preds = model.predict(X_val)
        
        # Calcoliamo il F1 Macro per misurare le performance bilanciate su tutte le classi
        macro_f1 = f1_score(y_val, preds, average='macro')
        fold_f1_scores.append(macro_f1)
        
    # Restituiamo la media dei Macro F1-score sui 5 fold
    return np.mean(fold_f1_scores)

print("[INFO] Inizio ottimizzazione degli iperparametri tramite Optuna con SMOTE+Tomek...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=15, timeout=400, show_progress_bar=True)

print("\n[INFO] Ottimizzazione completata!")
print(f"Miglior F1 Macro medio ottenuto: {study.best_value:.4f}")
print("Migliore configurazione di iperparametri trovata:")
for param, value in study.best_params.items():
    print(f" - {param}: {value}")

### 4. Addestramento Finale ed Esportazione (Cella 4)

Addestriamo il modello XGBoost definitivo sull'intero dataset utilizzando i migliori iperparametri individuati da Optuna. Configureremo l'obiettivo su `'multi:softprob'` in modo da supportare l'estrazione probabilistica e salveremo il modello in formato nativo `xgboost_model.json` per garantirne la portabilità.

In [ ]:
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier
from imblearn.combine import SMOTETomek
import joblib

# 1. Configurazione dei migliori iperparametri di XGBoost trovati da Optuna
best_xgb_params = study.best_params.copy()
best_xgb_params['objective'] = 'multi:softprob'
best_xgb_params['num_class'] = 5
best_xgb_params['eval_metric'] = 'mlogloss'
best_xgb_params['random_state'] = 42
best_xgb_params['n_jobs'] = -1

xgb_model = xgb.XGBClassifier(**best_xgb_params)

# 2. Configurazione di LightGBM e CatBoost con ottimi parametri di default
lgb_model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=5,
    random_state=42,
    n_jobs=-1,
    n_estimators=200,
    learning_rate=0.05,
    verbose=-1
)

cat_model = CatBoostClassifier(
    loss_function='MultiClass',
    random_state=42,
    thread_count=-1,
    iterations=250,
    learning_rate=0.06,
    verbose=0
)

# 3. Creazione del Voting Classifier (Soft Voting per predizioni probabilistiche)
voting_ensemble = VotingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgb', lgb_model),
        ('cat', cat_model)
    ],
    voting='soft'
)

# 4. Bilanciamento dell'intero dataset con SMOTE + Tomek prima del fit finale
print("[INFO] Applicazione di SMOTE + Tomek sul dataset completo per l'addestramento finale...")
smt_final = SMOTETomek(random_state=42)
X_resampled, y_resampled = smt_final.fit_resample(X, y)
print(f"[INFO] Shape dataset originario: X {X.shape}, y {y.shape} | Dataset bilanciato: X_res {X_resampled.shape}, y_res {y_resampled.shape}")

# 5. Addestramento dell'Ensemble 'Tridente' sul dataset bilanciato
print("[INFO] Addestramento del Voting Classifier Ensemble (XGBoost + LightGBM + CatBoost)...")
voting_ensemble.fit(X_resampled, y_resampled)
print("[INFO] Ensemble Voting Classifier addestrato con successo.")

# 6. Esportazione e salvataggio dell'ensemble su file locale (.pkl)
ensemble_filename = 'ensemble_pegi_model.pkl'
joblib.dump(voting_ensemble, ensemble_filename)
print(f"[INFO] File di modello dell'ensemble salvato con successo: '{ensemble_filename}'")

# Salvataggio dei nomi delle feature attese dal modello per allineamento
feature_names = X.columns.tolist()
joblib.dump(feature_names, 'model_features.pkl')
print(f"[INFO] Lista dei nomi delle feature salvata correttamente in 'model_features.pkl'.")

# 7. Test di caricamento di controllo per validare l'integrità dell'esportazione
try:
    control_model = joblib.load(ensemble_filename)
    print("[INFO] Controllo di caricamento riuscito! L'ensemble è pronto per l'uso in produzione.")
except Exception as e:
    print(f"[ERROR] Errore riscontrato nel tentativo di ricaricare l'ensemble: {e}")

### 5. Valutazione Metriche dell'Esame (Cella 5)

Generiamo e mostriamo a schermo i report di valutazione del modello:
- **Classification Report**: Precision, Recall e F1-score per ogni classe PEGI.
- **Matrice di Confusione Normalizzata**: Visualizzata graficamente tramite heatmap con percentuali per comprendere le aree di sovrapposizione.
- **Feature Importance**: Grafico orizzontale dell'importanza delle feature per quantificare il peso delle feature cromatiche LAB rispetto ai metadati di genere e prezzo.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Generazione delle predizioni sul dataset con l'ensemble Voting Classifier
y_pred = voting_ensemble.predict(X)

# Mappatura etichette testuali PEGI corrispondenti alle classi
pegi_labels = [f"PEGI {k}" for k in sorted(pegi_mapping.keys())]

# 2. Classification Report completo
print("="*65)
print("            REPORT DI VALUTAZIONE FINALE DELL'ENSEMBLE")
print("="*65)
print(classification_report(y, y_pred, target_names=pegi_labels))
print("="*65)

# 3. Matrice di Confusione Normalizzata in percentuale
cm = confusion_matrix(y, y_pred)
# Normalizzazione delle righe dividendo per la somma orizzontale e moltiplicando per 100
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_normalized, 
    annot=True, 
    fmt=".1f", 
    cmap="Blues", 
    xticklabels=pegi_labels, 
    yticklabels=pegi_labels,
    cbar_kws={'label': 'Frequenza delle Predizioni (%)'}
)
plt.title('Matrice di Confusione Normalizzata Ensemble (%)', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Classe Reale (True Label)', fontsize=12)
plt.xlabel('Classe Predetta (Predicted Label)', fontsize=12)
plt.tight_layout()
plt.show()

# 4. Grafico della Feature Importance (estratta dal miglior stimatore singolo dell'ensemble, XGBoost)
importance_scores = voting_ensemble.named_estimators_['xgb'].feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importance_scores
}).sort_values(by='Importance', ascending=False)

# Limitiamo la visualizzazione grafica alle 20 feature più significative per motivi di leggibilità
top_n = min(20, len(importance_df))
plt.figure(figsize=(12, 8))
sns.barplot(
    data=importance_df.head(top_n),
    x='Importance',
    y='Feature',
    palette='viridis',
    hue='Feature',
    legend=False
)
plt.title(f'Feature Importance (Top {top_n} estratte dal sub-modello XGBoost)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Valore di Importanza (Gini Gain)', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

# 5. Analisi del contributo pesato per tipologia di feature
grid_features = [col for col in X.columns if col.startswith('grid_')]
lbp_features = [col for col in X.columns if col.startswith('lbp_hist_')]
luminosity_features = [col for col in X.columns if col.startswith('L_')]
tfidf_features = [col for col in X.columns if col.startswith('tfidf_')]
genres_features = [col for col in X.columns if col.startswith('Is_')]

importance_grid = importance_df[importance_df['Feature'].isin(grid_features)]['Importance'].sum()
importance_lbp = importance_df[importance_df['Feature'].isin(lbp_features)]['Importance'].sum()
importance_lum = importance_df[importance_df['Feature'].isin(luminosity_features)]['Importance'].sum()
importance_tfidf = importance_df[importance_df['Feature'].isin(tfidf_features)]['Importance'].sum()
importance_genres = importance_df[importance_df['Feature'].isin(genres_features)]['Importance'].sum()
importance_price = importance_df[importance_df['Feature'] == 'price']['Importance'].sum() if 'price' in X.columns else 0.0

print("\n" + "="*65)
print("           ANALISI COMPARATIVA DEL CONTRIBUTO DELLE NUOVE FEATURE")
print("="*65)
print(f" - Spatial Grid Clustering (3x3):         {importance_grid:.4f} ({importance_grid*100:.1f}%)")
print(f" - Texture LBP (Uniform Hist):            {importance_lbp:.4f} ({importance_lbp*100:.1f}%)")
print(f" - Luminosità & Contrasto (Mean/Std L):   {importance_lum:.4f} ({importance_lum*100:.1f}%)")
print(f" - Text Mining (TF-IDF Key Vocabulary):   {importance_tfidf:.4f} ({importance_tfidf*100:.1f}%)")
print(f" - Metadati del Genere (Is_Action, ...):   {importance_genres:.4f} ({importance_genres*100:.1f}%)")
if 'price' in X.columns:
    print(f" - Metadato Prezzo (price):               {importance_price:.4f} ({importance_price*100:.1f}%)")
print("="*65)